In [106]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score


print(f"numpy version : {np.__version__}")
print(f"Pandas version {pd.__version__}f")
print(f"seaborn version: {sns.__version__}")
print(f"sklearn version {sklearn.__version__}")

numpy version : 2.0.2
Pandas version 2.2.2f
seaborn version: 0.13.2
sklearn version 1.6.1


In [107]:

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import os
import joblib

In [112]:


class MlPipeline:

  def __init__(self):
    self.scaler = None
    self.model = None
    self.feature_names = None

  def load_split(self,data,target_col,test_size=0.2,random_state=42):
    print(f"Loading data from {data}")
    df = pd.read_csv(data)

    df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)
    print(f"data shape")
    print(df.shape)

    # create independent and dependant feature
    X = df.drop(target_col,axis=1)
    y = df[target_col]

    # splite data
    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=test_size,random_state=random_state)

    print(f"Train data shape {X_train.shape}")
    print(f"Test data shape {X_test.shape}")

    return X_train,X_test,y_train,y_test


  def engineer_features(self,X_train,X_test):
    print("Engineering features")
    X_train = X_train.copy()
    X_test = X_test.copy()

    for col in X_train.select_dtypes(include =[np.number]).columns:
      if X_train[col].isnull().any():
        X_train[col].fillna(X_train[col].mean(),inplace=True)
        X_test[col].fillna(X_train[col].mean(),inplace=True)

    # one hot encoding
    X_train = pd.get_dummies(X_train,columns=['Sex','Embarked'],drop_first=True)
    X_test = pd.get_dummies(X_test,columns=['Sex','Embarked'],drop_first=True)


    return X_train,X_test


  def preprocess(self,X_train,X_test):
    self.scaler = StandardScaler()

    # Store original column names and indices before scaling
    self.feature_names = X_train.columns
    original_index_train = X_train.index
    original_index_test = X_test.index

    X_train_scaled_array = self.scaler.fit_transform(X_train)
    X_test_scaled_array = self.scaler.transform(X_test)

    # to convert back to dataframe
    X_train_scaled = pd.DataFrame(
        X_train_scaled_array,
        columns = self.feature_names,
        index = original_index_train
    )
    X_test_scaled = pd.DataFrame(
        X_test_scaled_array,
        columns = self.feature_names,
        index = original_index_test
    )

    return X_train_scaled ,X_test_scaled


  def train_model(self,X_train,y_train):

    self.model = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,

        n_jobs = 1,
        class_weight = "balanced"
    )

    cv_score = cross_val_score(
        self.model,X_train,y_train,cv=5,scoring="roc_auc"

    )
    print(f"cross-validation AUC: {cv_score.mean():.3f} (+/- {cv_score.std():.3f})")
    self.model.fit(X_train,y_train)

    return self.model

  def evaluation(self,X_test,y_test):
    y_pred = self.model.predict(X_test)
    y_pred_proba = self.model.predict_proba(X_test)[:,1]

    print("Test set evaluation:")
    print(classification_report(y_test, y_pred))
    test_auc = roc_auc_score(y_test, y_pred_proba)
    print(f"ROC-AUC: {test_auc:.3f}")

    return {
        'test_auc': test_auc
    }

  def save (self,filepath='models/model.pkl'):
    os.makedirs(os.path.dirname(filepath),exist_ok=True)

    artifacts = {
        'model':self.model,
        'scaler':self.scaler,
        'feature_names':self.feature_names
    }
    joblib.dump(artifacts,filepath)
    print(f"Model saved at {filepath}")



# usage
if __name__ == "__main__":
  pipeline = MlPipeline()


  #1 load and split
  X_train,X_test,y_train,y_test = pipeline.load_split(data="/content/drive/MyDrive/Synoris Training/Day2_training/Data_Pipeline/Titanic_Data.csv", target_col="Survived")

  #2 feature engineer
  X_train,X_test = pipeline.engineer_features(X_train,X_test)

  #3 preprocess
  X_train,X_test = pipeline.preprocess(X_train,X_test)

  #4 model traing
  pipeline.train_model(X_train,y_train)

  #5 Evalate
  pipeline.evaluation(X_test,y_test)

  #6 save
  pipeline.save('models/model_v1.pkl')

Loading data from /content/drive/MyDrive/Synoris Training/Day2_training/Data_Pipeline/Titanic_Data.csv
data shape
(418, 8)
Train data shape (334, 7)
Test data shape (84, 7)
Engineering features


/tmp/ipykernel_766/3960755137.py:36: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train[col].fillna(X_train[col].mean(),inplace=True)
/tmp/ipykernel_766/3960755137.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)',

cross-validation AUC: 1.000 (+/- 0.000)
Test set evaluation:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       1.00      1.00      1.00        34

    accuracy                           1.00        84
   macro avg       1.00      1.00      1.00        84
weighted avg       1.00      1.00      1.00        84

ROC-AUC: 1.000
Model saved at models/model_v1.pkl


## from diff method for understanding


In [84]:
df = pd.read_csv("/content/drive/MyDrive/Synoris Training/Day2_training/Data_Pipeline/Titanic_Data.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [44]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [58]:
df.nunique()

,0
Survived,2
Pclass,3
Sex,2
Age,79
SibSp,7
Parch,8
Fare,169
Embarked,3


In [45]:
# train_test_splite only input columns

X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=42)

In [46]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
336,2,male,32.0,0,0,13.0000,S
31,2,male,24.0,2,0,31.5000,S
84,2,male,NaN,0,0,10.7083,Q
287,1,male,24.0,1,0,82.2667,S
317,2,male,19.0,0,0,10.5000,S


In [47]:
y_train.head()

,Survived
336,0
31,0
84,0
287,0
317,0


In [ ]:
# we use index values for the columns because when data tranfer from numpy so that are in form of array so in pipline there was a mismatch

In [49]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# to hadle missing values we use imputer
# imputation tranformer

trf1 = ColumnTransformer([
    ('impute_age',SimpleImputer(),[2]),   # 2 is the index value of age
    ('impute_embarked',SimpleImputer(strategy='most_frequent'),[6])
],remainder='passthrough')

In [54]:
from sklearn.preprocessing import OneHotEncoder

# one hot encoding
trf2 = ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6])
],remainder='passthrough')

In [59]:
from sklearn.preprocessing import MinMaxScaler

# scailing
trf3 = ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10))  # apply on all columns slice(0,8)
])

In [61]:
from sklearn.feature_selection import SelectKBest, chi2

# feature selection
trf4 = SelectKBest(score_func=chi2,k=8)

In [64]:
from sklearn.ensemble import RandomForestClassifier

trf5 = RandomForestClassifier()

# create pipeline

In [74]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5)
])

In [73]:
# display pipeline
from sklearn import set_config
set_config(display='diagram')

In [76]:
pipe.fit(X_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x78e6abb97920>)),
                ('trf5', RandomForestClassifier())])

In [72]:
y_pred = pipe.predict(X_test)

In [77]:
pipe.score(X_test,y_test)

0.5952380952380952